In [1]:
import json
import re
from pypdf import PdfReader
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from groq import Groq

In [ ]:
import os
groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
def call_groq(prompt):
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user","content": prompt}],
        temperature=0.2)
    return response.choices[0].message.content

In [4]:
with open("/nlp/TBP NLP/dosen_sinta.json",
    "r",
    encoding="utf-8"
) as f:
    data_dosen = json.load(f)
print("Jumlah dosen:", len(data_dosen))

Jumlah dosen: 8


In [5]:
def build_dosen_documents(data_dosen):
    docs = []
    bidang_manual = {
        "MUHAMMAD FAHMY NADHIF": [
            "Explainable Artificial Intelligence",
            "Deep Learning",
            "Forecasting",
            "Machine Learning"
        ]}
    for dosen in data_dosen:
        nama = dosen.get(
            "nama_dosen",
            "")
        bidang = dosen.get(
            "bidang_keahlian",
            [])
        publikasi = []
        for _, daftar in dosen.get(
            "daftar_publikasi",
            {}).items():
            for artikel in daftar:
                judul = artikel.get(
                    "judul",
                    "")
                if judul:
                    publikasi.append(judul)
        penelitian = []
        for p in dosen.get(
            "daftar_penelitian",
            []):
            judul = p.get("judul", "")
            if judul:
                penelitian.append(judul)
        if len(bidang) == 0:
            if nama in bidang_manual:
                bidang = bidang_manual[nama]
        content = f"""
Nama Dosen:
{nama}
Bidang Keahlian:
{', '.join(bidang)}
Publikasi:
{' | '.join(publikasi)}
Penelitian:
{' | '.join(penelitian)}
"""
        docs.append(
            Document(
                page_content=content,
                metadata={
                    "nama_dosen":
                        nama,
                    "bidang_keahlian":
                        bidang,
                    "publikasi_text":
                        " ".join(publikasi),
                    "penelitian_text":
                        " ".join(penelitian)
                }
            )
        )
    return docs

In [6]:
dosen_docs = build_dosen_documents(data_dosen)
print(len(dosen_docs))

8


In [7]:
embedding_model = (HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"))
print("Embedding loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding loaded


In [8]:
dosen_store = QdrantVectorStore.from_documents(documents=dosen_docs, embedding=embedding_model, 
                                               location=":memory:", collection_name="dosen")

In [9]:
reader = PdfReader("/NLP/TBP NLP/Kurikulum Sadat.pdf")
full_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        full_text += "\n" + text

In [10]:
start = full_text.find("DESKRIPSI MATA KULIAH")
kurikulum_text = full_text[start:]

In [11]:
pattern = r'(?=\n\d+\.\s+[A-ZA-Za-z])'
sections = re.split(pattern,kurikulum_text)
sections = [
s.strip()
    for s in sections
    if "CPMK" in s
]
for s in sections[:3]:
    print(s[:1000])
    print("="*50)

1.    Pendidikan Agama Islam 
CPMK1 : Membimbing mahasiswa memiliki akhlak karimah (jujur, amanah, kerja keras, 
tanggung jawab, dan disiplin)  
CPMK2 : Mampu menerapkan akhlakul karimah dalam kehidupan sehari -hari, baik di 
kampus, keluarga, maupun masyarakat, serta mampu membangun hubungan 
harmonis dan saling menghormati dalam keragaman. 
 
Matakuliah Agama Islam merupakan Matakuliah Pengembangan Kepribadian (MPK) 
yang mengkaji ajaran Islam sebagai sumber nilai dan pedoman yang mengantarkan 
mahasiswa dalam pengembangan profesi dan kepribadian Islami. Setelah mengikuti 
matakuliah Agama Islam, mahasiswa dapat terbina keimanan dan ketakwaannya, 
berilmu pengetahuan dan berakhlak mulia serta menjadikan ajaran Islam sebagai 
landasan berpikir dan berperilaku dalam pengembangan profesi.
2.    Dasar Pemrograman 
CPMK1 : Mahasiswa bisa mengkoding menggunakan struktur, tipe data, operator, input 
dan output  
CPMK2 : Mahasiswa bisa  mengkoding statistika deskriptif mengguna kan struktur 

In [12]:
def get_matkul(text):
    match = re.search(r'(\d+)\.\s+(.*)', text)
    if match:
        return match.group(2).strip()
    return "Unknown"

In [13]:
kurikulum_docs = []
for section in sections:
    nama = get_matkul(section)
    kurikulum_docs.append(
        Document(
            page_content=section,
            metadata={
                "mata_kuliah": nama
            }
        )
    )
print(len(kurikulum_docs))

34


In [14]:
kurikulum_store = QdrantVectorStore.from_documents(
    documents=kurikulum_docs,
    embedding=embedding_model,
    location=":memory:",
    collection_name="kurikulum"
)

In [15]:
def similarity(text1, text2):
    emb1 = embedding_model.embed_query(text1)
    emb2 = embedding_model.embed_query(text2)
    return cosine_similarity([emb1],[emb2])[0][0]

In [40]:
def retrieve_dosen_ranked(query, k=5):
    results = dosen_store.similarity_search_with_score(
        query,
        k=len(dosen_docs)
    )
    ranked = []
    for doc, retrieval_score in results:
        expertise_score = 0
        for skill in doc.metadata.get(
            "bidang_keahlian",
            []
        ):
            sim = similarity(query, skill)
            expertise_score = max(expertise_score, sim)
        publication_text = (doc.metadata.get("publikasi_text", "")
            + " " +
            doc.metadata.get("penelitian_text", ""))
        if publication_text.strip():
            publication_score = similarity(query, publication_text)
        else:
            publication_score = expertise_score
        final_score = (0.35 * retrieval_score + 0.40 * expertise_score + 0.25 * publication_score)
        ranked.append({
            "nama":
                doc.metadata["nama_dosen"],
            "retrieval":
                retrieval_score,
            "expertise":
                expertise_score,
            "publication":
                publication_score,
            "final":
                final_score,
            "document":
                doc
        })
    ranked = sorted(
        ranked,
        key=lambda x: x["final"],
        reverse=True
    )
    return ranked[:k]

In [17]:
def expand_query(query):
    prompt = f"""
Anda adalah pakar Sains Data.
Topik:
{query}
Tugas:
Ekstrak 5-10 kata kunci akademik yang relevan.
Aturan:
- gunakan istilah mata kuliah
- gunakan istilah bidang ilmu
- pisahkan dengan koma
- jangan menjelaskan
Contoh:
big data, predictive analytics, machine learning
"""
    return call_groq(prompt)

In [18]:
def retrieve_kurikulum(query, k=3):
    expanded = expand_query(query)
    query_baru = (query + " " + expanded)
    hasil = kurikulum_store.similarity_search_with_score(query_baru, k=k)
    return hasil

In [19]:
def cek_kurikulum(query):
    hasil = retrieve_kurikulum(query, k=3)
    skor = []
    for _, score in hasil:
        skor.append(score)
    if len(skor) == 0:
        return 0
    return max(skor)

In [20]:
def get_kurikulum_context(query):
    hasil = retrieve_kurikulum(query, k=2)
    text = ""
    for doc, score in hasil:
        text += f"""
MATA KULIAH:
{doc.metadata['mata_kuliah']}
ISI:
{doc.page_content}
SKOR:
{round(score,3)}
"""
    return text

In [21]:
def status_kurikulum(score):
    if score >= 0.40:
        return "Sangat Sesuai"
    elif score >= 0.30:
        return "Sesuai"
    elif score >= 0.20:
        return "Perlu Revisi"
    else:
        return "Tidak Sesuai"

In [ ]:
def cek_domain_sains_data(query):
    prompt = f"""
Tentukan apakah topik berikut termasuk bidang Sains Data.
Topik:
{query}
Aturan:
- Data Science
- Machine Learning
- Deep Learning
- AI
- NLP
- Data Mining
- Big Data
- Computer Vision
- Forecasting
- Data Engineering
- Business Intelligence
Jawab hanya:
YA
atau
TIDAK
"""
    hasil = call_groq(prompt)
    return "YA" in hasil.upper()

In [ ]:
def analisis_rag(query):
    is_sd = cek_domain_sains_data(query)
    if not is_sd:
        return {
            "query": query,
            "status": "Tidak Sesuai",
            "skor": 0,
            "kurikulum": []
        }
    expanded = expand_query(query)
    query_baru = query + " " + expanded
    kurikulum = retrieve_kurikulum(query_baru)
    skor = max(score for _, score in kurikulum)
    if skor < 0.20:
        status = "Perlu Revisi"
    else:
        status = status_kurikulum(skor)
    hasil = {
        "query": query,
        "expanded_query": expanded,
        "status": status,
        "skor": skor,
        "kurikulum": kurikulum
    }
    dosen = retrieve_dosen_ranked(query)
    hasil["utama"] = dosen[0]
    hasil["alternatif"] = dosen[1:3]
    return hasil

In [ ]:
def build_prompt(hasil):
    if "utama" not in hasil:
        return f"""
# ROLE
Anda adalah evaluator akademik Program Studi Sains Data.
# TASK
Evaluasi apakah topik sesuai dengan kurikulum Program Studi Sains Data.
# CONTEXT
Topik:
{hasil['query']}
Status:
{hasil['status']}
# OUTPUT
1. Status kelayakan.
2. Kesesuaian kurikulum.
3. Alasan ketidaksesuaian.
# CONSTRAINT
- Gunakan hanya informasi pada dokumen.
- Jangan menggunakan pengetahuan umum.
- Jangan memberikan rekomendasi dosen.
"""
    utama = hasil["utama"]
    prompt = f"""
# ROLE
Anda adalah sistem rekomendasi dosen pembimbing skripsi Program Studi Sains Data.
# TASK
1. Menilai kesesuaian topik dengan kurikulum.
2. Menentukan dosen pembimbing utama.
3. Menentukan dua dosen alternatif.
4. Menjelaskan alasan berdasarkan data.
5. Memberikan saran penyempurnaan judul.
# TOPIK
{hasil['query']}
# STATUS
{hasil['status']}
"""
    if "expanded_query" in hasil:
        prompt += f"""
# KATA KUNCI AKADEMIK
{hasil['expanded_query']}
"""
    prompt += f"""
DOSEN UTAMA
Nama:
{utama['nama']}
Bidang Keahlian:
{utama['document'].metadata['bidang_keahlian']}
Bidang Inferensi:
{utama['document'].metadata.get('bidang_inferensi', '')}
Publikasi:
{utama['document'].metadata['publikasi_text'][:700]}
Penelitian:
{utama['document'].metadata['penelitian_text'][:700]}
DOSEN ALTERNATIF
"""
    for d in hasil["alternatif"]:
        prompt += f"""
Nama:
{d['nama']}
Skor Relevansi:
{round(d['final'],3)}
Bidang Keahlian:
{d['document'].metadata['bidang_keahlian']}
Publikasi:
{d['document'].metadata['publikasi_text'][:500]}
Penelitian:
{d['document'].metadata['penelitian_text'][:500]}
"""
    prompt += """
KURIKULUM
"""
    for doc, score in hasil["kurikulum"]:
        mk = doc.metadata.get("mata_kuliah", "Tidak diketahui")
        prompt += f"""
Mata Kuliah:
{mk}
Skor Similaritas:
{round(score,3)}
Isi Dokumen:
{doc.page_content}
"""
    prompt += """
# OUTPUT FORMAT
## 1. STATUS KELAYAKAN TOPIK
Tuliskan status kelayakan topik.
---
## 2. KESESUAIAN DENGAN KURIKULUM
Untuk maksimal 3 mata kuliah yang paling relevan:
- Nama mata kuliah.
- CPMK yang ditemukan.
- Kutipan dokumen.
- Hubungan topik dengan CPMK.
- Jelaskan alasan keterkaitan.
---
## 3. DOSEN PEMBIMBING UTAMA
Berikan satu dosen pembimbing utama.
Jelaskan berdasarkan:
- bidang keahlian,
- publikasi,
- penelitian.
- alasan kenapa menjadi pembimbing utama
---
## 4. DOSEN ALTERNATIF
Berikan dua dosen alternatif.
Untuk setiap dosen:
- bidang yang mendukung,
- publikasi yang relevan,
- penelitian yang relevan,
- keterbatasan.
---
## 5. SARAN PENYEMPURNAAN JUDUL
Berikan 2-3 alternatif judul dan jelaskan mengapa judul tersebut disarankan
# CONSTRAINT
1. Gunakan HANYA informasi pada data.
2. Jangan menggunakan pengetahuan umum.
3. Jangan menambahkan gelar akademik.
4. Jangan menambahkan jabatan.
5. Jangan membuat bidang keahlian baru.
6. Jangan membuat publikasi baru.
7. Jangan membuat penelitian baru.
8. Nama dosen harus sama persis dengan data.
9. Kesesuaian kurikulum WAJIB berasal dari dokumen kurikulum.
10. Sebutkan CPMK hanya jika ditemukan.
11. Jika informasi tidak tersedia tuliskan:
    "Tidak ditemukan informasi pendukung."
12. Prioritaskan mata kuliah yang paling relevan secara semantik.
13. Jangan memilih mata kuliah hanya karena kesamaan kata.
14. Relevansi dosen ditentukan oleh kedekatan topik, bukan jumlah publikasi.
15. Publikasi yang sedikit tidak boleh dirugikan apabila sangat relevan.
16. Fokus pada relevansi, bukan kuantitas publikasi.
17. Jika publikasi atau penelitian tidak relevan, tuliskan:
    "Tidak ditemukan informasi pendukung."
18. Jangan menghubungkan topik dengan penelitian yang tidak memiliki hubungan semantik yang jelas.
"""
    return prompt

In [26]:
def analisis_skripsi(query):
    hasil = analisis_rag(query)
    prompt = build_prompt(hasil)
    return call_groq(prompt)

In [45]:
hasil = analisis_skripsi("jadi aku mau buat penelitian perancangan pipeline prediksi menggunakan arsitektur lambda")
print(hasil)

## 1. STATUS KELAYAKAN TOPIK
Sangat Sesuai

## 2. KESESUAIAN DENGAN KURIKULUM
- Nama mata kuliah: Infrastruktur dan Platform Big Data
- CPMK yang ditemukan: CPMK1, CPMK2
- Kutipan dokumen: "Mahasiswa dapat memahami bagaimana data dikumpulkan, diolah dan disitribusikan oleh platform Big Data", "Mahasiswa dapat merancang sebuah server Big Data dengan kebutuhan infrasitruktur secara mandiri"
- Hubungan topik dengan CPMK: Topik "perancangan pipeline prediksi menggunakan arsitektur lambda" sangat terkait dengan CPMK1 dan CPMK2 karena memerlukan pemahaman tentang bagaimana data dikumpulkan, diolah, dan disitribusikan, serta kemampuan untuk merancang server Big Data.
- Jelaskan alasan keterkaitan: Alasan keterkaitan adalah karena topik tersebut memerlukan pemahaman tentang infrastruktur dan platform Big Data, yang merupakan fokus dari mata kuliah ini.

- Nama mata kuliah: Metode Visualisasi Data
- CPMK yang ditemukan: CPMK1, CPMK2, CPMK3
- Kutipan dokumen: "Mampu membangun model referensi vis